Detecteur de Déchets 

In [1]:
import tensorflow as tf
import os
import numpy as np

# Paramètres exacts déduits de README.roboflow.txt et data.yaml
IMG_SIZE = 640 # Les images sont déjà redimensionnées en 640x640 par Roboflow
NUM_CLASSES = 11
MAX_BOXES = 5 # Fixé arbitrairement, à ajuster si une image contient plus de 5 objets
BATCH_SIZE = 16

CLASS_NAMES = [
    'cheops', 'debris', 'double_start', 'earth_observation_sat_1', 
    'lisa_pathfinder', 'proba_2', 'proba_3_csc', 'proba_3_ocs', 
    'smart_1', 'soho', 'xmm_newton'
]

def parse_yolo_label(label_path):
    """
    Lit l'étiquette YOLO (ex: '10 0.719... 0.859... 0.491... 0.281...')
    et convertit [class, x_center, y_center, width, height]
    en [y_min, x_min, y_max, x_max, class_id] pour TensorFlow.
    """
    try:
        with open(label_path.numpy().decode('utf-8'), 'r') as f:
            lines = f.readlines()
    except Exception:
        lines = [] # Gestion des images sans boîte englobante

    boxes = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_id = float(parts[0]) # Ex: 10.0 pour xmm_newton
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            # Transformation mathématique des coordonnées normalisées
            x_min = x_center - (width / 2.0)
            y_min = y_center - (height / 2.0)
            x_max = x_center + (width / 2.0)
            y_max = y_center + (height / 2.0)

            boxes.append([y_min, x_min, y_max, x_max, cls_id])
    
    # Padding obligatoire pour que TensorFlow puisse batcher les tenseurs
    while len(boxes) < MAX_BOXES:
        boxes.append([0.0, 0.0, 0.0, 0.0, -1.0]) # Classe -1 = masque/padding
    
    boxes = boxes[:MAX_BOXES]
    return np.array(boxes, dtype=np.float32)

def tf_parse_label(label_path):
    return tf.py_function(parse_yolo_label, [label_path], tf.float32)

I0000 00:00:1791884907.641960   93195 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791884908.737093   93195 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791884912.561336   93195 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
def load_image_and_labels(image_path):
    # 1. Lecture et normalisation
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    # L'image fait déjà 640x640, mais on sécurise la dimension du tenseur
    img = tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])
    img = img / 255
    
    # 2. Remplacement du chemin (ex: ../train/images/... -> ../train/labels/...)
    label_path = tf.strings.regex_replace(image_path, "images", "labels")
    label_path = tf.strings.regex_replace(label_path, "\.jpg", ".txt")
    
    # 3. Parsing
    boxes = tf_parse_label(label_path)
    boxes.set_shape([MAX_BOXES, 5])
    
    return img, boxes

def create_dataset(image_dir):
    # Pointer vers le dossier spécifique (train, valid, ou test)
    file_pattern = os.path.join(image_dir, "*.jpg")
    dataset = tf.data.Dataset.list_files(file_pattern, shuffle=False)
    
    # Shuffle uniquement pour le train pour éviter de biaiser l'évaluation
    if "train" in image_dir:
        dataset = dataset.shuffle(buffer_size=1000)
        
    dataset = dataset.map(load_image_and_labels, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Instanciation
train_ds = create_dataset("data/train/images")
val_ds = create_dataset("data/valid/images")
test_ds = create_dataset("data/test/images")

<>:11: SyntaxWarning: invalid escape sequence '\.'
<>:11: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_93195/2584566578.py:11: SyntaxWarning: invalid escape sequence '\.'
  label_path = tf.strings.regex_replace(label_path, "\.jpg", ".txt")
<unknown>:11: SyntaxWarning: invalid escape sequence '\.'
